# Credit Risk Classification with RFX-Fuse

This example demonstrates RFX-Fuse's complete ML pipeline on the Kaggle
[Give Me Some Credit](https://www.kaggle.com/c/GiveMeSomeCredit) dataset
(150,000 borrowers, binary default prediction).

**What this notebook covers:**

1. **Missing-data imputation** with `rfx_impute_rough` (iterative RF imputation)
2. **Feature engineering** for credit risk
3. **Classification** with `RandomForestClassifier` (500 trees)
4. **Model evaluation** (precision-recall, threshold tuning)
5. **RFX-Fuse 6-in-1 capabilities** from a single trained model:
   - Overall feature importance (Gini / permutation)
   - Overall similarity (proximity) importance
   - Local feature importance per observation
   - Top-K most similar borrowers
   - Local proximity importance (why they are similar)
   - Outlier detection

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
import RFXFuse as rfx
from rfx_impute import rfx_impute_rough
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    classification_report,
)

## 2. Load Data

Download the dataset from [Kaggle](https://www.kaggle.com/c/GiveMeSomeCredit/data)
and place `cs-training.csv` in this directory (or update the path below).

In [2]:
dat = pd.read_csv("cs-training.csv")

In [3]:
dat.head()

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [4]:
dat.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 12 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   Unnamed: 0                            150000 non-null  int64  
 1   SeriousDlqin2yrs                      150000 non-null  int64  
 2   RevolvingUtilizationOfUnsecuredLines  150000 non-null  float64
 3   age                                   150000 non-null  int64  
 4   NumberOfTime30-59DaysPastDueNotWorse  150000 non-null  int64  
 5   DebtRatio                             150000 non-null  float64
 6   MonthlyIncome                         120269 non-null  float64
 7   NumberOfOpenCreditLinesAndLoans       150000 non-null  int64  
 8   NumberOfTimes90DaysLate               150000 non-null  int64  
 9   NumberRealEstateLoansOrLines          150000 non-null  int64  
 10  NumberOfTime60-89DaysPastDueNotWorse  150000 non-null  int64  
 11  NumberOfDep

## 3. Exploratory Data Analysis & Cleaning

The dataset has 150K rows and 11 features. Key observations:
- **Class imbalance**: ~6.7% positive (defaulted) vs 93.3% negative
- **Missing values**: `MonthlyIncome` (19.8%) and `NumberOfDependents` (2.6%)
- **Sentinel values**: Delinquency columns use 96 and 98 as special codes (not real counts)


In [6]:
dat = dat.drop(columns="Unnamed: 0")

In [7]:
dat.describe()

,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
count,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,1.202690e+05,150000.000000,150000.000000,150000.000000,150000.000000,146076.000000
mean,0.066840,6.048438,52.295207,0.421033,353.005076,6.670221e+03,8.452760,0.265973,1.018240,0.240387,0.757222
std,0.249746,249.755371,14.771866,4.192781,2037.818523,1.438467e+04,5.145951,4.169304,1.129771,4.155179,1.115086
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.029867,41.000000,0.000000,0.175074,3.400000e+03,5.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.154181,52.000000,0.000000,0.366508,5.400000e+03,8.000000,0.000000,1.000000,0.000000,0.000000
75%,0.000000,0.559046,63.000000,0.000000,0.868254,8.249000e+03,11.000000,0.000000,2.000000,0.000000,1.000000
max,1.000000,50708.000000,109.000000,98.000000,329664.000000,3.008750e+06,58.000000,98.000000,54.000000,98.000000,20.000000


In [8]:
delinquency_cols = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]

In [9]:
dat[delinquency_cols] = dat[delinquency_cols].replace({96: np.nan, 98: np.nan})

In [10]:
dat.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 11 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   SeriousDlqin2yrs                      150000 non-null  int64  
 1   RevolvingUtilizationOfUnsecuredLines  150000 non-null  float64
 2   age                                   150000 non-null  int64  
 3   NumberOfTime30-59DaysPastDueNotWorse  149731 non-null  float64
 4   DebtRatio                             150000 non-null  float64
 5   MonthlyIncome                         120269 non-null  float64
 6   NumberOfOpenCreditLinesAndLoans       150000 non-null  int64  
 7   NumberOfTimes90DaysLate               149731 non-null  float64
 8   NumberRealEstateLoansOrLines          150000 non-null  int64  
 9   NumberOfTime60-89DaysPastDueNotWorse  149731 non-null  float64
 10  NumberOfDependents                    146076 non-null  float64
dtypes: float64(

In [11]:
dat['SeriousDlqin2yrs'].value_counts()

SeriousDlqin2yrs
0    139974
1     10026
Name: count, dtype: int64

## 4. Imputation with RFX-Fuse

Five features have missing values. `MonthlyIncome` is the worst at ~20%.
We use `rfx_impute_rough` — an iterative Random Forest imputation method that
trains an RF on observed values to predict each missing feature in turn.


In [ ]:
dat.isna().sum() / dat.shape[0]

In [13]:
print(f"Total missing: {dat.isna().sum().sum():,}")

,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,0.766127,45,2.0,0.802982,9120.0,13,0.0,6,0.0,2.0
1,0,0.957151,40,0.0,0.121876,2600.0,4,0.0,0,0.0,1.0
2,0,0.658180,38,1.0,0.085113,3042.0,2,1.0,0,0.0,0.0
3,0,0.233810,30,0.0,0.036050,3300.0,5,0.0,0,0.0,0.0
4,0,0.907239,49,1.0,0.024926,63588.0,7,0.0,1,0.0,0.0


In [14]:
rf_data = rfx_impute_rough(
    np.array(dat), n_iterations=5, n_trees=50, use_gpu=False, seed=42
)

RFX Imputation (rough)
  Samples: 150,000, Features: 11
  Total missing: 34,462 (2.1%)
  Features with missing: 5
  Categorical features: 1, Numeric: 10
  Initial imputation: median/mode (rough)

  Iteration 1/5
Training Random Forest Regressor with 50 trees...

💻 CPU MEMORY INFORMATION
📊 System Memory:
   Total: 15.3 GB
   Available: 13.7 GB
   Used: 1.6 GB
   Usage: 10.4%

    Feature 3 (num): imputed 269 values, mean change=0.0869
Training Random Forest Regressor with 50 trees...

💻 CPU MEMORY INFORMATION
📊 System Memory:
   Total: 15.3 GB
   Available: 13.7 GB
   Used: 1.6 GB
   Usage: 10.7%

    Feature 7 (num): imputed 269 values, mean change=0.5690
Training Random Forest Regressor with 50 trees...

💻 CPU MEMORY INFORMATION
📊 System Memory:
   Total: 15.3 GB
   Available: 13.6 GB
   Used: 1.7 GB
   Usage: 10.8%

    Feature 9 (num): imputed 269 values, mean change=0.1989
Training Random Forest Regressor with 50 trees...

💻 CPU MEMORY INFORMATION
📊 System Memory:
   Total: 15.3 GB

In [15]:
rf_dat = pd.DataFrame(rf_data[0], columns=dat.columns)

## 5. Feature Engineering

Create domain-specific features for credit risk modeling:
- **income_per_dep**: monthly income divided by number of dependents
- **monthly_debt**: debt-to-income applied to monthly income
- **num_unsecured_lines**: open credit lines minus real estate loans
- **total_delinquencies**: sum of all delinquency counts
- **age_buckets**: discretized age groups


In [ ]:
rf_dat["DebtRatio"] = rf_dat["DebtRatio"].clip(upper=2)
rf_dat["income_per_dep"] = rf_dat["MonthlyIncome"] / (rf_dat["NumberOfDependents"] + 1)
rf_dat["monthly_debt"] = rf_dat["MonthlyIncome"] * rf_dat["DebtRatio"]
rf_dat["num_unsecured_lines"] = (
    rf_dat["NumberOfOpenCreditLinesAndLoans"] - rf_dat["NumberRealEstateLoansOrLines"]
)
rf_dat["total_delinquencies"] = rf_dat[delinquency_cols].sum(axis=1)

In [17]:
for col in rf_dat.columns:
    upper = rf_dat[col].quantile(0.99)
    lower = rf_dat[col].quantile(0.01)
    rf_dat[col] = rf_dat[col].clip(lower=lower, upper=upper)

/home/cjk/coding_practice-venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:4596: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a


In [44]:
rf_dat["age_buckets"] = pd.cut(
    rf_dat["age"], bins=[0, 40, 75, 100], labels=["young", "middle-aged", "elderly"]
)
rf_dat = pd.get_dummies(rf_dat, dtype=int)

In [24]:
print(f"Final shape: {rf_dat.shape}")
rf_dat.head()

'SeriousDlqin2yrs'

In [25]:
assert rf_dat.isna().sum().sum() == 0, "Still have missing values!"

SeriousDlqin2yrs                        0
RevolvingUtilizationOfUnsecuredLines    0
age                                     0
NumberOfTime30-59DaysPastDueNotWorse    0
DebtRatio                               0
MonthlyIncome                           0
NumberOfOpenCreditLinesAndLoans         0
NumberOfTimes90DaysLate                 0
NumberRealEstateLoansOrLines            0
NumberOfTime60-89DaysPastDueNotWorse    0
NumberOfDependents                      0
income_per_dep                          0
monthly_debt                            0
num_unsecured_lines                     0
total_delinquencies                     0
age_buckets_young                       0
age_buckets_middle-aged                 0
age_buckets_elderly                     0
dtype: int64

In [45]:
## 6. Train / Test Split & Model Training


In [46]:
X = rf_dat.drop(columns=["SeriousDlqin2yrs"])
y = rf_dat["SeriousDlqin2yrs"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]:,} samples  |  Test: {X_test.shape[0]:,} samples")
print(f"Default rate: {y_train.mean():.1%}")

57836     0.0
132895    0.0
27981     0.0
37852     0.0
103813    0.0
         ... 
18048     0.0
3895      0.0
109980    0.0
74354     0.0
80530     0.0
Name: SeriousDlqin2yrs, Length: 120000, dtype: float32

In [47]:
rf = rfx.RandomForestClassifier(
    use_gpu=False,
    ntree=500,
    mtry=5,
    minndsize=5,
    iseed=50,
    compute_importance=True,
    compute_local_importance=True,
    compute_proximity_importance=True,
    compute_leaf_assignments=True,
)

In [48]:
rf.fit(np.array(X_train), np.array(y_train))

57836     0.0
132895    0.0
27981     0.0
37852     0.0
103813    0.0
         ... 
18048     0.0
3895      0.0
109980    0.0
74354     0.0
80530     0.0
Name: SeriousDlqin2yrs, Length: 120000, dtype: float32

## 7. Model Evaluation

With 6.7% default rate, accuracy alone is misleading. We focus on
**Average Precision (AUPRC)** and tune the classification threshold
using the F1-optimal point on the precision-recall curve.


In [ ]:
probs = rf.predict_proba(np.array(X_test, dtype=np.float64))
auprc = average_precision_score(y_test, probs[:, 1])
print(f"Average Precision (AUPRC): {auprc:.4f}")

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, probs[:, 1])
f1 = 2 * precision * recall / (precision + recall + 1e-15)
best_thresh = thresholds[np.argmax(f1)]
print(f"F1-optimal threshold: {best_thresh:.3f}")

preds = (probs[:, 1] >= best_thresh).astype(int)
print(classification_report(y_test, preds))

## 8. RFX-Fuse 6-in-1: Post-Training Capabilities

With a single trained model, RFX-Fuse gives you six capabilities that
typically require separate libraries (SHAP, FAISS, isolation forest, etc.).

### 8a. Extract all outputs


In [ ]:
over_imp = rf.feature_importances_()
local_imp = rf.get_local_importance()
over_prox_imp = rf.get_overall_proximity_importance()
local_prox_imp = rf.get_local_proximity_importance()
outliers = rf.compute_outliers(mode="greedy")

### 8b. Overall Feature Importance

Which features are most important for predicting default?


In [ ]:
pd.DataFrame({
    "feature": X_train.columns,
    "importance": over_imp,
}).sort_values("importance", ascending=False)

### 8c. Overall Similarity (Proximity) Importance

Which features drive borrower similarity? This tells you what the model
uses to group people together — useful for fair lending and segmentation.


pd.DataFrame({
    "feature": X_train.columns,
    "proximity_importance": over_prox_imp,
}).sort_values("proximity_importance", ascending=False)

### 8d. Local Feature Importance (per observation)

Like SHAP values, but computed natively by the RF proximity structure.
Here we look at a specific borrower to see which features drove their prediction.


In [ ]:
sample_idx = 18
pd.DataFrame({
    "feature": X_train.columns,
    "local_importance": local_imp[sample_idx],
}).sort_values("local_importance", ascending=False)

### 8e. Top-K Most Similar Borrowers

Find the 10 most similar borrowers to a given individual based on RF proximity.
This is useful for ECOA adverse action explanations and peer-group analysis.



In [ ]:
similar_indices = rf.get_top_k_similar(sample_idx, k=10)
print(f"Top 10 borrowers most similar to borrower {sample_idx}:")
print(similar_indices)

similar_df = X_train.iloc[similar_indices]
similar_df

### 8f. Local Proximity Importance (why are they similar?)

For a given borrower, which features explain why their nearest neighbors
are considered similar? This goes beyond global importance and provides
observation-level explanations of similarity.


In [ ]:
pd.DataFrame({
    "feature": X_train.columns,
    "local_proximity_importance": local_prox_imp[sample_idx],
}).sort_values("local_proximity_importance", ascending=False)

### 8g. Outlier Detection

Leverage proximity-based outlier scores. Observations that are far from
all other observations (low proximity to everything) receive high outlier scores.


In [ ]:
outlier_scores = pd.Series(outliers, name="outlier_score")
print(f"Top 20 most outlier-like borrowers (highest scores):")
outlier_scores.nlargest(20)

In [ ]:
top_outlier = outlier_scores.idxmax()
print(f"\nMost outlier-like borrower (index {top_outlier}):")
X_train.iloc[top_outlier]

## 9. Summary

From a **single trained model**, RFX-Fuse delivered:

| # | Capability | Method |
|---|-----------|--------|
| 1 | Overall feature importance | `feature_importances_()` |
| 2 | Overall similarity importance | `get_overall_proximity_importance()` |
| 3 | Local feature importance | `get_local_importance()` |
| 4 | Top-K most similar observations | `get_top_k_similar()` |
| 5 | Local proximity importance | `get_local_proximity_importance()` |
| 6 | Outlier detection | `compute_outliers()` |

All six outputs come at no additional model-fitting cost. Together they cover
feature attribution, peer-group analysis, ECOA adverse-action explanations,
and anomaly detection — capabilities that typically require separate specialized tools.
